In [1]:
# ============================================================
# PHASE 1.2B — ESM-2 SLIDING-WINDOW EMBEDDING
# CPU-compatible version
# Model: facebook/esm2_t6_8M_UR50D
# ============================================================
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", None)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.2.0+cpu
CUDA available: False


In [2]:
!pip install "protobuf<4.21" --upgrade

In [3]:
# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path.cwd()

# Thư mục gốc chứa toàn bộ mã nguồn/mô hình (lùi lại 1 cấp: ../modeling)
MODEL_DIR_ROOT = BASE_DIR.parent

# Trỏ đến thư mục phase 1.1 để lấy dữ liệu splits
PHASE_1_1_DIR = MODEL_DIR_ROOT / "phase1_protein_baseline"
SPLIT_DIR = PHASE_1_1_DIR / "splits"

# Các thư mục đầu ra nằm ngay bên trong thư mục hiện tại (BASE_DIR)
EMBED_DIR = BASE_DIR / "embeddings"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

# Tự động tạo các thư mục con nếu chưa tồn tại
for folder in [EMBED_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Base dir:", BASE_DIR)
print("Split dir:", SPLIT_DIR)

Base dir: d:\khoane\MAHE\Project\model\phase1_2_esm2_sliding_window_embedding
Split dir: d:\khoane\MAHE\Project\model\phase1_protein_baseline\splits


In [4]:
# ============================================================
# LOAD SAVED SPLITS
# ============================================================

train_df = pd.read_csv(SPLIT_DIR / "train_protein_v1.csv")
val_df = pd.read_csv(SPLIT_DIR / "val_protein_v1.csv")
test_df = pd.read_csv(SPLIT_DIR / "test_protein_v1.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Train: (1274, 35)
Validation: (273, 35)
Test: (273, 35)

Train labels:
label
0    637
1    637
Name: count, dtype: int64

Validation labels:
label
1    137
0    136
Name: count, dtype: int64

Test labels:
label
0    137
1    136
Name: count, dtype: int64


In [5]:
# ============================================================
# CLEAN SEQUENCES
# ============================================================

STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AAS])
    return seq


for split_df in [train_df, val_df, test_df]:
    split_df["sequence_clean"] = split_df["sequence"].apply(clean_sequence)
    split_df["sequence_clean_length"] = split_df["sequence_clean"].str.len()

print("Train sequence length summary:")
display(train_df["sequence_clean_length"].describe())

print("Validation sequence length summary:")
display(val_df["sequence_clean_length"].describe())

print("Test sequence length summary:")
display(test_df["sequence_clean_length"].describe())

Train sequence length summary:


count    1274.000000
mean      744.512559
std       643.266085
min        41.000000
25%       354.000000
50%       555.000000
75%       926.000000
max      7388.000000
Name: sequence_clean_length, dtype: float64

Validation sequence length summary:


count      273.000000
mean       869.728938
std       2114.423720
min         56.000000
25%        370.000000
50%        576.000000
75%        977.000000
max      34350.000000
Name: sequence_clean_length, dtype: float64

Test sequence length summary:


count     273.000000
mean      774.432234
std       689.578849
min        51.000000
25%       326.000000
50%       574.000000
75%      1035.000000
max      4861.000000
Name: sequence_clean_length, dtype: float64

In [6]:
# ============================================================
# LOAD ESM-2 MODEL
# ============================================================

ESM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
print("Loading:", ESM_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)
esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME)

esm_model.to(device)
esm_model.eval()

print("Model loaded.")

Using device: cpu
Loading: facebook/esm2_t6_8M_UR50D



Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded.


In [7]:
# ============================================================
# SLIDING-WINDOW CONFIG
# ============================================================

WINDOW_SIZE = 1022
STRIDE = 1022          # non-overlapping windows for first CPU-compatible run
SAVE_EVERY = 50        # save checkpoint every 50 proteins

REPRESENTATION_NAME = "ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022"

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Representation:", REPRESENTATION_NAME)

Window size: 1022
Stride: 1022
Representation: ESM2_t6_8M_sliding_window_mean_pool_1022_stride_1022
